# Looker

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/looker/looker.ipynb)

Looker (Google Cloud) is a BI and embedded-analytics platform whose semantic layer, LookML, is version-controlled code. Looker stores no data: it compiles a user's field picks into SQL and runs it in your warehouse. This notebook covers LookML modeling, fan-out and symmetric aggregates, PDTs, caching and datagroups, aggregate awareness, security and operations, how Looker differs from Data Studio (formerly Looker Studio), and how other BI tools reuse the LookML semantic layer, with the core mechanics reproduced in Python and DuckDB.

*Last reviewed: November 2026, against the Looker documentation as of November 2026.*

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"duckdb": "duckdb>=1.0", "pandas": "pandas>=2.2"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

## Looker is a SQL compiler

The LookML model is the program; a user's chosen fields, filters and pivots are the input; generated SQL is the output, executed by the warehouse. Consequences:

+ The warehouse bill (bytes scanned, slots, credits) is the real cost driver, and warehouse performance is Looker performance.
+ Every number is defined once, in code, and reviewed in Git. A metric change is a diff.
+ Looker does not transform or ingest data. Heavy transformation belongs upstream (dbt, warehouse marts); Looker models the result.

The object hierarchy:

```text
project     Git repository of .lkml files
  model     connection + which explores exist (acme.model.lkml)
    explore query surface: a base view plus joins; what users start from
      view  one table or derived table: dimensions (GROUP BY) and measures (aggregates)
```

A typical view and explore:

```lookml
view: orders {
  sql_table_name: analytics.fct_orders ;;

  dimension: order_id {
    primary_key: yes
    type: number
    sql: ${TABLE}.order_id ;;
  }
  dimension_group: created {
    type: time
    timeframes: [date, week, month, year]
    sql: ${TABLE}.created_at ;;
  }
  dimension: status { type: string  sql: ${TABLE}.status ;; }
  measure: order_count { type: count }
  measure: total_revenue {
    type: sum
    sql: ${TABLE}.amount ;;
    value_format_name: usd
  }
  measure: average_order_value {
    type: number
    sql: ${total_revenue} / NULLIF(${order_count}, 0) ;;
  }
}

explore: orders {
  join: customers {
    type: left_outer
    relationship: many_to_one
    sql_on: ${orders.customer_id} = ${customers.customer_id} ;;
  }
}
```

The next cell is a toy compiler with the same shape: it reads a model dictionary and emits the SQL Looker would generate for a field selection. Dimensions become `GROUP BY` columns; measures are aggregated; joins are added only when a selected field needs them (Looker prunes unused joins too).

### Where you'll use it

- **Governed metrics:** a revenue definition written once in LookML is reused by every dashboard, so two teams cannot report different numbers.

In [ ]:
import duckdb

con = duckdb.connect()
con.execute("""
CREATE TABLE fct_orders AS SELECT * FROM (VALUES
    (1, 10, TIMESTAMP '2026-01-05 10:00', 'complete', 40000),
    (2, 10, TIMESTAMP '2026-01-20 12:00', 'complete', 60000),
    (3, 20, TIMESTAMP '2026-02-02 09:00', 'returned', 50000),
    (4, 30, TIMESTAMP '2026-02-14 15:00', 'complete', 25000)
) t(order_id, customer_id, created_at, status, amount_cents);
CREATE TABLE dim_customers AS SELECT * FROM (VALUES
    (10, 'enterprise', 'EMEA'), (20, 'smb', 'US'), (30, 'smb', 'EMEA')
) t(customer_id, segment, region);
CREATE TABLE fct_order_items AS SELECT * FROM (VALUES
    (1, 'A'), (1, 'B'), (1, 'C'), (2, 'A'), (2, 'B'), (2, 'C'), (3, 'A'), (4, 'D')
) t(order_id, sku);
""")

MODEL = {
    "orders": {
        "table": "fct_orders",
        "dimensions": {
            "created_month": "DATE_TRUNC('month', orders.created_at)",
            "status": "orders.status",
        },
        "measures": {
            "order_count": "COUNT(*)",
            "total_revenue": "SUM(orders.amount_cents) / 100.0",
        },
    },
    "customers": {
        "table": "dim_customers",
        "join_on": "orders.customer_id = customers.customer_id",
        "dimensions": {"segment": "customers.segment", "region": "customers.region"},
        "measures": {},
    },
}


def compile_query(fields: list[str], filters: dict[str, str] | None = None) -> str:
    """Return the SQL a Looker explore on `orders` would generate for the fields."""
    filters = filters or {}
    select, group_by, views = [], [], {"orders"}
    for position, field in enumerate(fields, 1):
        view, name = field.split(".")
        views.add(view)
        if name in MODEL[view]["dimensions"]:
            select.append(f"{MODEL[view]['dimensions'][name]} AS {view}_{name}")
            group_by.append(str(position))
        else:
            select.append(f"{MODEL[view]['measures'][name]} AS {view}_{name}")
    where = []
    for field, value in filters.items():
        view, name = field.split(".")
        views.add(view)
        where.append(f"{MODEL[view]['dimensions'][name]} = '{value}'")
    sql = f"SELECT {', '.join(select)}\nFROM {MODEL['orders']['table']} AS orders"
    for view in sorted(views - {"orders"}):
        sql += f"\nLEFT JOIN {MODEL[view]['table']} AS {view} ON {MODEL[view]['join_on']}"
    if where:
        sql += "\nWHERE " + " AND ".join(where)
    if group_by:
        sql += "\nGROUP BY " + ", ".join(group_by)
    return sql + "\nORDER BY 1"


sql = compile_query(["orders.created_month", "orders.total_revenue"])
print(sql)
print(con.sql(sql).df())
assert "JOIN" not in sql  # customers is not needed, so the join is pruned

sql = compile_query(["customers.segment", "orders.order_count"], filters={"customers.region": "EMEA"})
print(sql)
print(con.sql(sql).df())

## Looker, Data Studio, and the semantic layer outside Looker

**Looker is not Looker Studio.** Google sells two different products, and the naming caused years of confusion. In April 2026 Google retired the "Looker Studio" name and brought back **Data Studio** for its free, self-service dashboard tool (with a paid **Data Studio Pro** tier for team administration). Interview answers and architecture docs should keep them apart:

| | Looker | Data Studio (formerly Looker Studio) |
|---|---|---|
| What it is | enterprise BI and embedded-analytics platform | lightweight report and dashboard builder |
| Semantic layer | LookML: version-controlled dimensions, measures, joins, access filters | none of its own; fields are defined per data source |
| Who builds | developers write LookML in Git; business users explore | analysts and marketers connect sources and drag charts |
| Governance | one reviewed metric definition, row-level `access_filter`, content permissions | sharing per report; definitions can drift between reports |
| Cost | licensed platform | free, with a paid Pro tier |

**Using the LookML semantic layer from other tools.** The **Open SQL Interface** exposes LookML models over JDBC: each model appears as a schema, each Explore as a table, and each field as a column named `view.field`. Any JDBC client can query it, and Google ships BI connectors on top of it, starting with Tableau (Desktop and Server); partners such as ThoughtSpot use the same interface. Constraints worth stating:

+ The model's connection must be BigQuery, and the user needs the `explore` permission on the model.
+ Only `SELECT` statements in GoogleSQL; no joins, window functions or subqueries, because the Explore already defines the joins.
+ Measures are wrapped in `AGGREGATE()` so Looker applies the LookML aggregation (including symmetric aggregates) instead of the client re-aggregating.

```sql
-- A Tableau or JDBC client querying the `orders` Explore in the `acme` model.
SELECT `orders.status`, AGGREGATE(`orders.total_revenue`) AS total_revenue
FROM `acme`.`orders`
GROUP BY `orders.status`;
```

Queries arriving this way show up in System Activity with the source `sql_interface`, so you can see which governed metrics other tools consume. The design point: define a metric once in LookML and let Looker, Tableau and spreadsheets all read the same definition, rather than rebuilding it in each tool.

## Dimensions, measures and calculations

+ **Dimension types**: `string`, `number`, `yesno`, `tier`, `date`/`time` via `dimension_group` (one declaration yields `created_date`, `created_month`, ...), `location`, `zipcode`. Always declare a `primary_key`: symmetric aggregates depend on it.
+ **Measure types** are SQL decisions: `sum`, `average`, `count`, `count_distinct`, `min`, `max`, `median`, `percentile`, and `number` for ratios built from other measures. A ratio must be built from summed components (`${total_margin} / NULLIF(${total_revenue}, 0)`), never as an average of per-row percentages.
+ **Filtered measures**: `filters: [status: "complete"]` turns into `SUM(CASE WHEN status = 'complete' THEN amount END)`.

Three layers of calculation:

| Layer | Runs where | Reusable | Use for |
|---|---|---|---|
| LookML measure / dimension | Warehouse SQL | Everywhere, governed | Anything more than one person needs |
| Custom field / custom measure (Explore UI) | Warehouse SQL | Saved with that query | Ad-hoc analysis |
| Table calculation (`running_total(${orders.total_revenue})`) | Looker, on the returned result | That tile only | Running totals, percent of previous, pivots post-query |

Table calculations operate on the result set only: they cannot see rows beyond the row limit, and they are not filters on the source.

## Fan-out and symmetric aggregates

Joining a one-to-many relationship repeats the "one" side's rows. A plain `SUM(orders.amount)` across `orders -> order_items` counts each order once per item, silently. Looker defends against this with **symmetric aggregates**: when the joined explore contains a one-to-many (or many-to-many) join and the measure's view has a `primary_key`, Looker rewrites `SUM` and `AVG` so each primary key contributes once:

```sql
SUM(DISTINCT big_hash(pk) * 1e6 + amount) - SUM(DISTINCT big_hash(pk) * 1e6)
```

Adding a large unique offset per primary key makes duplicate rows identical, `DISTINCT` collapses them, and subtracting the offsets leaves the true sum. The cell below runs the same trick in DuckDB with exact `HUGEINT` arithmetic.

Requirements and costs:

+ The `relationship:` on every join must be correct. `many_to_one` declared on what is really `one_to_many` disables the protection.
+ The `primary_key` must be unique. A non-unique key silently undercounts.
+ `count_distinct` survives fan-out naturally; `sum`, `average` and `count` need symmetric aggregates.
+ Symmetric aggregates are slower (hashing, `DISTINCT` over big numbers). For heavy explores, pre-aggregate the child to the parent grain in a derived table instead.

### Where you'll use it

- **Finance reports:** joining orders to payments without counting an order twice when it has several payments.

In [ ]:
naive = con.sql("""
    SELECT SUM(o.amount_cents) / 100.0
    FROM fct_orders o JOIN fct_order_items i USING (order_id)
""").fetchall()[0][0]

KEY_OFFSET = 10**12  # larger than any amount_cents, so key and value never collide

symmetric = con.sql(f"""
    SELECT (SUM(DISTINCT CAST(hash(o.order_id) AS HUGEINT) * {KEY_OFFSET} + o.amount_cents)
          - SUM(DISTINCT CAST(hash(o.order_id) AS HUGEINT) * {KEY_OFFSET})) / 100.0
    FROM fct_orders o JOIN fct_order_items i USING (order_id)
""").fetchall()[0][0]

truth = con.sql("SELECT SUM(amount_cents) / 100.0 FROM fct_orders").fetchall()[0][0]
item_count = con.sql("SELECT COUNT(*) FROM fct_order_items").fetchall()[0][0]
print(f"naive join sum={naive}  symmetric={symmetric}  truth={truth}  items={item_count}")
assert float(symmetric) == truth == 1750.0
assert naive > truth

## Explores: shaping the query surface

An explore is a product decision as much as a modeling one:

+ `fields: [ALL_FIELDS*, -orders.internal_cost]` curates the field picker; `hidden: yes` removes internal fields.
+ `always_filter` forces a filter users can change but not remove (`created_date: "last 90 days"`), which caps scan cost on partitioned tables.
+ `sql_always_where` adds a filter users cannot see or change (for example `deleted_at IS NULL`).
+ `conditionally_filter` requires one of several filters, with a default.
+ `access_filter` scopes rows by a user attribute (see security).
+ Join types: `left_outer` (default), `inner`, `full_outer`, `cross`. Relationships: `one_to_one`, `many_to_one`, `one_to_many`, `many_to_many`.

Model a many-to-many through a bridge table and let symmetric aggregates handle the measures, or pre-aggregate. Complex nested data (BigQuery `ARRAY`/`STRUCT`) is modeled by joining an `UNNEST` as a separate view with `relationship: one_to_many`. `extends` and refinements (`+view_name`) let you reuse and layer views without copying.

## Derived tables and PDTs

A **derived table** is a view defined by SQL (or by LookML `explore_source`, a "native derived table") instead of a physical table.

| Kind | Behaviour | Use for |
|---|---|---|
| Ephemeral derived table | Inlined as a CTE in every query | Light reshaping |
| Persistent derived table (PDT) | Materialized in a scratch schema, rebuilt on a trigger | Expensive rollups queried by many tiles |
| Incremental PDT | Appends or replaces only recent partitions (`increment_key`, `increment_offset`) | Large append-mostly event tables |
| Aggregate table (`aggregate_table` in an explore) | A PDT used automatically by aggregate awareness | Dashboard-grain rollups |

```lookml
datagroup: nightly_etl {
  sql_trigger: SELECT MAX(loaded_at) FROM analytics.etl_audit ;;
  max_cache_age: "24 hours"
}

view: customer_ltv {
  derived_table: {
    sql: SELECT customer_id, SUM(amount) AS lifetime_revenue, MIN(created_at) AS first_order_at
         FROM analytics.fct_orders GROUP BY 1 ;;
    datagroup_trigger: nightly_etl
    indexes: ["customer_id"]
  }
  dimension: customer_id { primary_key: yes  sql: ${TABLE}.customer_id ;; }
  dimension: lifetime_revenue { type: number  sql: ${TABLE}.lifetime_revenue ;; }
}
```

Persistence options: `datagroup_trigger` (rebuild when the trigger SQL's value changes; preferred), `sql_trigger_value` (the same per table), and `persist_for: "24 hours"` (rebuild on first query after expiry; unpredictable latency for the unlucky first user). The regenerator checks triggers on a schedule (default every five minutes), builds the new table under a new name, then swaps it in.

Design rules: a PDT is not a replacement for a dbt mart (it lives outside your transformation lineage and tests); keep PDTs for BI-specific rollups; key triggers to an ETL audit table rather than a clock; for big append-only facts use incremental PDTs with an offset that covers late-arriving data.

The next cell simulates a datagroup: the PDT and the query cache are rebuilt only when the trigger value changes.

### Where you'll use it

- **Heavy aggregations:** a persistent derived table pre-computes a daily summary that dashboards read instead of the raw events.

In [ ]:
con.execute("""
CREATE TABLE etl_audit AS SELECT TIMESTAMP '2026-03-01 02:00' AS loaded_at;
""")


class Datagroup:
    """Rebuilds a PDT and drops cached results only when the trigger value changes."""

    def __init__(self, trigger_sql: str, pdt_name: str, pdt_sql: str):
        self.trigger_sql, self.pdt_name, self.pdt_sql = trigger_sql, pdt_name, pdt_sql
        self.last_trigger = None
        self.builds = 0
        self.cache: dict[str, list] = {}
        self.warehouse_queries = 0

    def check_trigger(self) -> None:
        value = con.sql(self.trigger_sql).fetchall()[0][0]
        if value != self.last_trigger:
            con.execute(f"CREATE OR REPLACE TABLE {self.pdt_name} AS {self.pdt_sql}")
            self.last_trigger = value
            self.builds += 1
            self.cache.clear()

    def run(self, sql: str) -> list:
        if sql not in self.cache:  # Looker's cache key includes the generated SQL text
            self.warehouse_queries += 1
            self.cache[sql] = con.sql(sql).fetchall()
        return self.cache[sql]


nightly = Datagroup(
    "SELECT MAX(loaded_at) FROM etl_audit",
    "customer_ltv",
    "SELECT customer_id, SUM(amount_cents) / 100.0 AS lifetime_revenue FROM fct_orders GROUP BY 1",
)
tile_sql = "SELECT COUNT(*), SUM(lifetime_revenue) FROM customer_ltv"

for _ in range(3):  # regenerator checks every few minutes; nothing loaded in between
    nightly.check_trigger()
    for _viewer in range(10):
        nightly.run(tile_sql)
print(f"after day 1: builds={nightly.builds} warehouse_queries={nightly.warehouse_queries}")
assert (nightly.builds, nightly.warehouse_queries) == (1, 1)

con.execute("INSERT INTO fct_orders VALUES (5, 40, TIMESTAMP '2026-03-02 08:00', 'complete', 10000)")
con.execute("INSERT INTO etl_audit VALUES (TIMESTAMP '2026-03-02 02:00')")
nightly.check_trigger()
result = nightly.run(tile_sql)
print(f"after load:  builds={nightly.builds} warehouse_queries={nightly.warehouse_queries} result={result}")
assert nightly.builds == 2 and result[0][0] == 4

## Caching

Looker caches query results keyed by the generated SQL (which includes user-attribute values from access filters, so RLS users do not share entries). A cached result is served until it expires by the model's or explore's `persist_with` datagroup (`max_cache_age`) or the datagroup trigger fires. Default policy is one hour.

+ Tie caches to ETL with a datagroup: dashboards are then free between loads and refresh immediately after. Leaving the default one-hour policy on a nightly table wastes warehouse money, and a long `max_cache_age` on an hourly table shows stale data.
+ Dashboards whose tiles share an explore and filters share cache entries; 50 bespoke tiles mean 50 warehouse queries.
+ "Clear cache and refresh" bypasses the cache for that user; schedules can be set to run "when the datagroup triggers" so deliveries always use fresh data.

### Where you'll use it

- **Dashboard speed:** repeated loads of the same explore hit the cache, which cuts warehouse cost on busy dashboards.

In [ ]:
TILES = 12
VIEWS_PER_DAY = 400
COST_PER_QUERY_USD = 0.02
LOADS_PER_DAY = 1

uncached = TILES * VIEWS_PER_DAY
default_one_hour = TILES * 24            # worst case: one miss per tile per hour
etl_datagroup = TILES * LOADS_PER_DAY    # one miss per tile per load
for name, queries in [("no cache", uncached), ("1h policy", default_one_hour), ("ETL datagroup", etl_datagroup)]:
    print(f"{name:14s} {queries:5d} queries/day  ${queries * COST_PER_QUERY_USD:7.2f}/day")
assert etl_datagroup < default_one_hour < uncached

## Aggregate awareness

`aggregate_table` declarations inside an explore let Looker answer a query from a smaller rollup when the rollup contains every requested dimension and filter, at an equal or coarser time granularity, and every measure is re-aggregable (`sum`, `count`, `min`, `max`; `count_distinct` and `average` only at exactly matching grain). Users and dashboards query the same explore; Looker picks the smallest table that can answer.

```lookml
explore: orders {
  aggregate_table: monthly_by_status {
    query: {
      dimensions: [orders.created_month, orders.status]
      measures: [orders.total_revenue, orders.order_count]
    }
    materialization: { datagroup_trigger: nightly_etl }
  }
}
```

The selection rule, implemented and checked against the base table:

In [ ]:
con.execute("""
CREATE TABLE agg_monthly_by_status AS
SELECT DATE_TRUNC('month', created_at) AS created_month, status,
       SUM(amount_cents) AS amount_cents, COUNT(*) AS order_count
FROM fct_orders GROUP BY 1, 2
""")

AGGREGATE_TABLES = [
    {"name": "agg_monthly_by_status", "dimensions": {"created_month", "status"}, "rows": 4},
]
ADDITIVE = {"total_revenue", "order_count"}


def choose_source(dimensions: set[str], measures: set[str]) -> str:
    candidates = [
        table for table in AGGREGATE_TABLES
        if dimensions <= table["dimensions"] and measures <= ADDITIVE
    ]
    return min(candidates, key=lambda t: t["rows"])["name"] if candidates else "fct_orders"


source = choose_source({"created_month"}, {"total_revenue"})
from_agg = con.sql(f"""
    SELECT created_month, SUM(amount_cents) / 100.0 FROM {source} GROUP BY 1 ORDER BY 1
""").fetchall()
from_base = con.sql("""
    SELECT DATE_TRUNC('month', created_at), SUM(amount_cents) / 100.0 FROM fct_orders GROUP BY 1 ORDER BY 1
""").fetchall()
print("monthly revenue answered from:", source, from_agg)
assert source == "agg_monthly_by_status" and from_agg == from_base
assert choose_source({"created_month", "customer_id"}, {"total_revenue"}) == "fct_orders"

## Security and governance

Content access and data access are separate:

+ **Roles = permission set + model set.** Permission sets say what a user can do (`explore`, `see_sql`, `develop`, `download_without_limit`); model sets say which models they can do it on.
+ **Folders** control who sees saved Looks and dashboards.
+ **Row-level**: `access_filter` maps a field to a **user attribute**; Looker adds the predicate to every generated query, so it cannot be bypassed in the Explore UI. `sql_always_where` with `{{ _user_attributes['tenant_id'] }}` does the same in custom SQL.
+ **Column-level**: `required_access_grants` on fields, views, explores or joins hides them unless the user's attribute matches an `access_grant`.
+ **Masking**: Looker has no native masking engine; mask with a dimension whose SQL uses Liquid on a user attribute or access grant, or rely on warehouse masking policies with per-user OAuth connections.
+ **Connection credentials**: a service account by default; OAuth per user on BigQuery and Snowflake lets warehouse policies apply.

```lookml
access_grant: can_see_pii { user_attribute: department  allowed_values: ["finance", "compliance"] }

explore: orders {
  access_filter: { field: customers.region  user_attribute: sales_region }
}

dimension: email {
  required_access_grants: [can_see_pii]
  sql: ${TABLE}.email ;;
}
```

The access filter in action with the toy compiler: the same tile compiles to different SQL per user, and a user with no attribute value sees nothing.

### Where you'll use it

- **Access by team:** row-level access rules in a model show each regional team only its own customers.

In [ ]:
USER_ATTRIBUTES = {"ana": {"sales_region": "EMEA"}, "ben": {"sales_region": "US"}, "eve": {}}
NO_ACCESS = "__no_access__"


def query_as(user: str, fields: list[str]) -> list:
    region = USER_ATTRIBUTES[user].get("sales_region", NO_ACCESS)
    return con.sql(compile_query(fields, filters={"customers.region": region})).fetchall()


for user in USER_ATTRIBUTES:
    print(user, query_as(user, ["customers.region", "orders.order_count"]))
assert query_as("ana", ["customers.region", "orders.order_count"]) == [("EMEA", 3)]
assert query_as("eve", ["customers.region", "orders.order_count"]) == []

### Multi-tenancy and embedding

Common patterns for SaaS analytics: one model with `access_filter` on `tenant_id` (simplest, one cache namespace per attribute value); per-tenant connections or schemas chosen by Liquid (`sql_table_name: {{ _user_attributes['tenant_schema'] }}.orders ;;`) when data is physically separated; per-tenant folders and groups for content. **Signed embedding** has your backend create a short-lived signed URL (or use the cookieless embed API) containing the external user ID, models, permissions and user attributes; the attributes then drive the access filters, so tenant isolation is enforced in generated SQL rather than in the front end.

## Liquid templating

Liquid makes LookML dynamic: `{% parameter %}`, `{% condition %}` (applies a filter's value inside derived-table SQL), `{{ _user_attributes['x'] }}`, `{% if field._in_query %}`, and `{{ value }}` / `{{ rendered_value }}` in `html:` and `link:`.

```lookml
parameter: metric_selector {
  type: unquoted
  allowed_value: { label: "Revenue" value: "amount" }
  allowed_value: { label: "Margin"  value: "margin" }
}
measure: selected_metric {
  type: sum
  sql: ${TABLE}.{% parameter metric_selector %} ;;
}

view: filtered_events {
  derived_table: {
    sql: SELECT * FROM events WHERE {% condition event_date_filter %} event_date {% endcondition %} ;;
  }
  filter: event_date_filter { type: date }
}
```

`{% condition %}` pushes a user's filter into the inner query, so a derived table over a huge event table scans only the requested partitions.

## Development workflow

+ **Development Mode** gives each developer a personal Git branch and a sandboxed copy of PDTs; production users never see uncommitted changes.
+ **LookML validator** checks syntax and references; the **Content Validator** finds Looks and dashboards broken by renamed or removed fields (and can bulk-replace field names).
+ **Data tests** (`test:` blocks with `explore_source` and `assert`) run in CI or before deploy, for example "total revenue for 2025 equals the finance number" or "primary key is unique".
+ Deploy through a pull request to the production branch; advanced deploy mode lets you pin production to a specific commit.

```lookml
test: order_id_is_unique {
  explore_source: orders {
    column: order_id {}
    column: count { field: orders.order_count }
    sorts: [orders.order_count: desc]
    limit: 1
  }
  assert: order_id_is_unique { expression: ${orders.order_count} = 1 ;; }
}
```

Lineage and documentation: `description:` and `label:` on every field surface in the Explore UI; the API exposes which content uses which fields; dbt docs or a catalog (Dataplex, Atlan, DataHub) connect LookML fields back to warehouse tables.

## Scheduling, alerting, actions and the API

+ **Schedules** deliver Looks and dashboards (PDF, CSV, PNG) to email, Slack, S3, SFTP or webhooks, on a cron or **when a datagroup triggers** (the right choice after ETL).
+ **Alerts** evaluate a condition on a dashboard tile on a schedule and notify when it is met.
+ **Data actions** (`action:` on a field) and the Action Hub send a row's data to an external endpoint (create a ticket, update a CRM record).
+ **API 4.0 and SDKs** run inline queries, manage content and users, and trigger PDT builds. Orchestrators use them to rebuild PDTs or send deliveries right after a load:

```python
import time
import looker_sdk

sdk = looker_sdk.init40()  # reads LOOKERSDK_BASE_URL, LOOKERSDK_CLIENT_ID, LOOKERSDK_CLIENT_SECRET

def rebuild_pdt(model: str, view: str) -> None:
    build = sdk.start_pdt_build(model_name=model, view_name=view)
    while (state := sdk.check_pdt_build(build.materialization_id)).status not in {"complete", "error", "killed"}:
        time.sleep(10)
    if state.status != "complete":
        raise RuntimeError(f"PDT {view} build {state.status}: {state.message}")
```

**SQL Runner** runs raw SQL against a connection for debugging and prototyping derived tables. Its queries are unmodeled and ungoverned; promote anything useful into a view.

## Performance: diagnosing a slow query

1. Open the query in the Explore and read the **SQL** tab: unexpected joins, symmetric aggregates, missing partition filters?
2. Check **System Activity > Performance** and the query's timing breakdown: queued in Looker (connection limit reached), running in the warehouse, or post-processing (large result, table calculations, pivots)?
3. Use the warehouse's query profile for the generated SQL.

Fixes, roughly by impact: partition filters via `always_filter`; aggregate tables or PDTs for dashboard grains; pre-aggregate child tables instead of relying on symmetric aggregates; fewer tiles sharing explores; datagroup caching; raise the connection's max concurrent queries only if the warehouse has headroom.

## Real-time data and limitations

Looker can be "real-time" only as far as the warehouse is: live queries on streaming-ingested tables (BigQuery streaming, Snowflake Snowpipe) with a short or disabled cache, and dashboards set to auto-refresh. Limitations worth stating in an interview: cost depends on warehouse usage; LookML has a learning curve and needs developers; heavy visual customization is limited compared with Tableau; it is not an ETL tool; very high-cardinality ad-hoc exploration can be expensive.

## Practice scenarios

+ **Multi-brand marketing attribution**: one explore over a touchpoint fact, attribution weights in a dbt model, a `parameter` to switch attribution models, aggregate tables for the weekly channel dashboard.
+ **Embedded B2B SaaS analytics**: signed embed with `tenant_id` attribute, `access_filter` on every explore, per-tenant cache entries, and a data test that checks no explore lacks the filter.
+ **Regulated reporting**: `required_access_grants` on PII, deploys only through reviewed pull requests, data tests asserting finance totals, schedules triggered by the ETL datagroup.

Questions to check yourself:

1. Why does a wrong `relationship:` silently inflate revenue, and what two declarations fix it?
2. When is an incremental PDT unsafe without an `increment_offset`?
3. Why do access-filtered users get lower cache hit rates?
4. Which measure types can aggregate awareness roll up from a coarser table, and which cannot?

### Where you'll use it

- **Operational reports:** a scheduled look emails the daily backlog to the operations team and alerts when a value crosses a threshold.